# Lesson 13 - Rock Paper Scissors Vision

In this lesson, SpiderPi uses MediaPipe hand recognition to play rock, paper, scissors with you.

## What We Will Do

1. Warm up the robot and camera
2. Detect a hand gesture
3. Turn the gesture into `rock`, `paper`, or `scissors`
4. Let SpiderPi count down: rock, paper, scissors, go
5. Reveal the robot move with the claw
6. Act out the result with lights, speech, and movement

In [ ]:
from lesson_loader import setup
setup()
from student_robot_v2 import bot
import random

bot.help()


In [ ]:
# Warm up SpiderPi for the game
bot.display.text("RPS Time", "Show a hand", "Rock Paper", "Scissors")
bot.display.smile()
bot.lights.blue()
bot.speech.say("Let's play rock, paper, scissors. Show your hand to the camera.")
bot.camera.center_all()
bot.arm.home()
bot.arm.open()


In [ ]:
# Detect a hand and inspect the result
hand_result = bot.vision.recognize_hands(show=True)
print(hand_result)


In [ ]:
# Pick the first game move that looks like rock, paper, or scissors
possible_moves = hand_result.get("game_moves", [])
player_move = possible_moves[0] if possible_moves else None
print("Player move:", player_move)

if player_move is None:
    bot.lights.yellow()
    bot.display.text("Try Again", "I could not", "see rock", "paper scissors")
    bot.speech.say("I could not read that hand sign yet. Try again with a clear rock, paper, or scissors pose.")
else:
    bot.display.text("Player Move", player_move.title(), "Nice!", "My turn next")
    bot.speech.say(f"I think you played {player_move}.")


In [ ]:
# SpiderPi chooses a move, counts down, and decides who wins
robot_move = random.choice(["rock", "paper", "scissors"])
print("Robot move:", robot_move)

def show_robot_move(move):
    if move == "rock":
        bot.arm.close()
        bot.display.text("Spider Move", "Rock", "Claw closed", "Strong grip")
    elif move == "paper":
        bot.arm.open()
        bot.display.text("Spider Move", "Paper", "Claw open", "Flat hand")
    else:
        bot.arm.half_open()
        bot.display.text("Spider Move", "Scissors", "Half open", "Snip snip")

def countdown_and_show(move):
    beats = [
        ("Rock", 18),
        ("Paper", 15),
        ("Scissors", 12),
        ("Go", 9),
    ]
    bot.arm.open()
    for word, height in beats:
        bot.display.text("Countdown", word, "Arm drops", "Get ready")
        bot.speech.say(word)
        bot.arm.move(0, 15, height, seconds=0.35)
    show_robot_move(move)

countdown_and_show(robot_move)

def winner(player, robot):
    if player is None:
        return "retry"
    if player == robot:
        return "draw"
    wins = {
        ("rock", "scissors"),
        ("paper", "rock"),
        ("scissors", "paper"),
    }
    return "player" if (player, robot) in wins else "robot"

result = winner(player_move, robot_move)
print("Result:", result)


In [ ]:
# Make SpiderPi react to the game result
if result == "retry":
    bot.lights.yellow()
    bot.display.text("No Score", "Need a clear", "hand sign", "Try again")
    bot.speech.say("Let's try that round again.")
elif result == "draw":
    bot.lights.purple()
    bot.display.text("Draw", f"You: {player_move}", f"Me: {robot_move}", "Same move!")
    bot.body.twist()
    bot.speech.say("It's a draw. Great minds think alike.")
elif result == "player":
    bot.lights.red()
    bot.display.text("You Win", f"You: {player_move}", f"Me: {robot_move}", "Well played")
    bot.body.backward(0.4)
    bot.speech.say("You win this round. Nice move.")
else:
    bot.lights.green()
    bot.display.text("I Win", f"You: {player_move}", f"Me: {robot_move}", "SpiderPi wins")
    bot.body.dance()
    bot.speech.say("Spider Pi wins this round.")

bot.arm.home()
bot.body.stop()


## Challenge

Try one or more of these upgrades:

- change the countdown arm heights or speed
- make SpiderPi choose a special move sound for each result
- keep score across three rounds
- make different gestures trigger different action groups